**Dim User**\
**Batch read from Bronze**

In [0]:
df = spark.read.format("parquet") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimUser")

display(df.limit(5)) 


**Streaming ingestion Bronze → Silver (Auto Loader)-DimUser**

In [0]:
# Streaming read from Bronze (schemaLocation)
df_user = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/schema_checkpoint") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimUser")

**Transformations**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name"))) \
                 .drop("_rescued_data")


**Project path setup (Python utilities)**

In [0]:
import os
import sys

project_path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_path)

print(project_path)


**Apply Reusable Transformations (Drop Columns & Deduplicate)**

In [0]:
from utils.transformation import reusable

df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.drop_duplicates(['user_id'])


**Write DimUser stream to Silver table**

In [0]:
df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/stream_checkpoint") \
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/data") \
    .toTable("spotify_cata.silver.DimUser")


**Query DimUser Silver table**

In [0]:
df_user =  spark.read.table("spotify_cata.silver.DimUser")
display(df_user)

**Streaming write to Silver (checkpointLocation)**

In [0]:
"""" 
df_user.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/stream_checkpoint") \
    .trigger(availableNow=True) \
    .start("abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/data")
"""


**Preview the streaming DataFrame directly**

In [0]:
"""
display(
    df_user,
    checkpointLocation="abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/display_checkpoint",
    outputMode="append"
)
"""

**Read back from Silver after the write**

In [0]:
"""
df_silver = spark.read.format("delta") \
    .load("abfss://silver@samstorageazureproject.dfs.core.windows.net/DimUser/data")

display(df_silver.limit(10))
"""


**Dim Artist Batch read from Bronze**

In [0]:
df = spark.read.format("parquet") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimArtist")

display(df.limit(5)) 


**Streaming ingestion Bronze → Silver (Auto Loader)-DimArtist**

In [0]:
# Streaming read from Bronze (schemaLocation)
df_artist = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimArtist/schema_checkpoint") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimArtist")

**Transformation**

In [0]:
df_artist_obj = reusable()
df_artist = df_artist_obj.dropColumns(df_artist, ['_rescued_data'])
df_artist = df_artist.drop_duplicates(['artist_id'])

**Write DimArtist stream to Silver table**

In [0]:
df_artist.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimArtist/stream_checkpoint") \
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimArtist/data") \
    .toTable("spotify_cata.silver.DimArtist")


**Query DimArtist Silver table**

In [0]:
df_artist =  spark.read.table("spotify_cata.silver.DimArtist")
display(df_artist)

**Dim Track**\
**Batch read from Bronze**

In [0]:
df = spark.read.format("parquet") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimTrack")

display(df.limit(5)) 

**Streaming ingestion Bronze → Silver (Auto Loader)-DimTrack**

In [0]:
# Streaming read from Bronze (schemaLocation)
df_track = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimTrack/schema_checkpoint") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimTrack")

**Transformations**

In [0]:
# Add durationFlag column based on duration_sec
df_track = df_track.withColumn(
    "durationFlag",
    when(col("duration_sec") < 150, "low")      # less than 150 seconds → low
    .when(col("duration_sec") < 300, "medium")  # less than 300 seconds → medium
    .otherwise("high")                          # 300 seconds or more → high
)


In [0]:
# Replace hyphens with spaces in track_name
df_track = df_track.withColumn(
    "track_name",
    regexp_replace(col("track_name"), "-", " ")
)


In [0]:
# Drop the _rescued_data column from df_track
df_track = reusable().dropColumns(df_track, ['_rescued_data'])


**Write DimTrack stream to Silver table**

In [0]:
df_track.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimTrack/stream_checkpoint") \
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimTrack/data") \
    .toTable("spotify_cata.silver.DimTrack")


**Query DimArtist Silver table**

In [0]:
df_track =  spark.read.table("spotify_cata.silver.DimTrack")
display(df_track)

**Dim Date**\
**Batch read from Bronze**

In [0]:
df = spark.read.format("parquet") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimDate")

display(df.limit(5)) 


**Streaming ingestion Bronze → Silver (Auto Loader)-DimDate**


In [0]:
# Streaming read from Bronze (schemaLocation)
df_date = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimDate/schema_checkpoint") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/DimDate")

**Transformation**

In [0]:
# Drop the _rescued_data column from df_date
df_date = reusable().dropColumns(df_date, ['_rescued_data'])


**Write DimDate stream to Silver table**


In [0]:
df_date.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimDate/stream_checkpoint") \
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@samstorageazureproject.dfs.core.windows.net/DimDate/data") \
    .toTable("spotify_cata.silver.DimDate")


**Query DimDate silver table**

In [0]:
df_date =  spark.read.table("spotify_cata.silver.DimDate")
display(df_date)

**Fact Stream**\
**Batch read from Bronze**

In [0]:
df = spark.read.format("parquet") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/FactStream")

display(df.limit(5)) 


**Streaming ingestion Bronze → Silver (Auto Loader)-FactStream**


In [0]:
# Streaming read from Bronze (schemaLocation)
df_fact = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .option("cloudFiles.schemaLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/FactStream/schema_checkpoint") \
    .option("schemaEvolutionMode", "addNewColumns") \
    .load("abfss://bronze@samstorageazureproject.dfs.core.windows.net/FactStream")

In [0]:
# Drop the _rescued_data column from df_fact
df_fact = reusable().dropColumns(df_fact, ['_rescued_data'])


**Write Fact stream to Silver table**



In [0]:
df_fact.writeStream.format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@samstorageazureproject.dfs.core.windows.net/FactStream/stream_checkpoint") \
    .trigger(availableNow=True)\
    .option("path", "abfss://silver@samstorageazureproject.dfs.core.windows.net/FactStream/data") \
    .toTable("spotify_cata.silver.FactStream")


**Query FactStream silver table**

In [0]:
df_fact =  spark.read.table("spotify_cata.silver.FactStream")
display(df_fact)